# Neural Jigsaw Image Reconstruction Demo

This notebook is a clean portfolio/demo version of the jigsaw image reconstruction project.

The goal is to reconstruct a full **96×96 RGB image** from **nine scrambled and cropped 28×28 patches**. The model combines patch-level CNN encoding, attention-based patch reasoning, differentiable soft patch reassembly, and a U-Net-style reconstruction decoder.

Project result: **30/30**  
Final reported test MAE: **0.0292**


## 1. Setup

This notebook assumes the repository has this structure:

```text
neural-jigsaw-image-reconstruction/
├── README.md
├── requirements.txt
├── src/
│   ├── config.py
│   ├── data.py
│   ├── model.py
│   ├── train.py
│   ├── evaluate.py
│   ├── visualize.py
│   └── utils.py
└── notebooks/
    └── neural_jigsaw_reconstruction_demo.ipynb
```

If you run this notebook locally, install the dependencies first:

```bash
pip install -r requirements.txt
```


In [ ]:
import sys
from pathlib import Path

# Make sure the repository root is on the Python path.
# This works when the notebook is inside the notebooks/ folder.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(repo_root))

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from src.config import JigsawConfig
from src.model import build_model
from src.utils import print_model_stats

print("TensorFlow version:", tf.__version__)
print("Repository root:", repo_root)


## 2. Build the model

The model is built from the refactored `src/` files. It is initialized with a dummy input so that the parameter count can be checked immediately.


In [ ]:
cfg = JigsawConfig()
model = build_model(cfg, name="jigsaw_v11b_demo")

print_model_stats(model)


## 3. Architecture summary

The model has five main components:

1. **Patch encoder** — encodes each 28×28 patch into a feature vector.
2. **Edge encoder** — extracts boundary cues from patch edges.
3. **Attention blocks** — allow each patch to compare itself with the other patches.
4. **Soft patch reassembly** — predicts a differentiable 9×9 soft assignment.
5. **U-Net-style decoder** — reconstructs the final 96×96 RGB image and fills missing borders.


In [ ]:
model.summary()


## 4. Optional: load trained weights

The repository does not include large weight files by default. If you have the trained checkpoint locally, place it in the repository root and run the cell below.

Expected checkpoint name:

```text
best_jigsaw_v11b_lite_deeper.weights.h5
```


In [ ]:
weights_path = repo_root / cfg.best_weights_path

if weights_path.exists():
    model.load_weights(str(weights_path))
    print(f"Loaded weights from: {weights_path}")
else:
    print("No local weight file found.")
    print("The model is built correctly, but it is currently using random initialization.")


## 5. Patch extraction demo with a synthetic image

This cell shows the input format expected by the model: 9 scrambled patches of shape 28×28×3.

Here I use a synthetic image so that the notebook can run without downloading the STL-10 dataset.


In [ ]:
from src.data import extract_patches

# Create a synthetic 96x96 RGB image for demonstrating the patch pipeline.
yy, xx = np.mgrid[0:96, 0:96]
synthetic = np.zeros((96, 96, 3), dtype=np.float32)
synthetic[..., 0] = xx / 95.0
synthetic[..., 1] = yy / 95.0
synthetic[..., 2] = 0.5 * (np.sin(xx / 8.0) + 1.0)

patches_ordered = extract_patches(tf.constant(synthetic), cfg).numpy()

perm = np.random.permutation(9)
patches_scrambled = patches_ordered[perm]

print("Ordered patches shape:", patches_ordered.shape)
print("Scrambled patches shape:", patches_scrambled.shape)
print("Permutation:", perm)


In [ ]:
def make_patch_grid(patches):
    grid = patches.reshape(3, 3, cfg.crop_size, cfg.crop_size, cfg.channels)
    grid = np.transpose(grid, (0, 2, 1, 3, 4))
    return grid.reshape(3 * cfg.crop_size, 3 * cfg.crop_size, cfg.channels)

plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(synthetic)
plt.title("Original synthetic image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(make_patch_grid(patches_scrambled))
plt.title("Scrambled 28×28 patches")
plt.axis("off")

plt.tight_layout()
plt.show()


## 6. Reconstruction demo

If trained weights are available, this cell shows the model reconstruction. If no weights are loaded, the output is random and is only useful for checking that the pipeline runs.


In [ ]:
x = patches_scrambled[None].astype("float32")
reconstruction = model(x, training=False).numpy()[0]

plt.figure(figsize=(10, 3))

plt.subplot(1, 3, 1)
plt.imshow(make_patch_grid(patches_scrambled))
plt.title("Input patches")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(reconstruction)
plt.title("Model output")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(synthetic)
plt.title("Target image")
plt.axis("off")

plt.tight_layout()
plt.show()


## 7. Reported final results

The final submitted model was evaluated on the STL-10 based test split.

| Metric | Value |
|---|---:|
| Test MAE | 0.0292 |
| Test Std | 0.0379 |
| Patch-position accuracy | 88.3% |
| Improvement over baseline | 84.9% |
| Trainable parameters | 5.94M |
| Final grade | 30/30 |

These numbers are reported from the final trained checkpoint used in the course submission.


## 8. Notes

This notebook is meant as a **GitHub demo notebook**. The full training workflow is implemented in:

```text
src/train.py
```

The evaluation utilities are implemented in:

```text
src/evaluate.py
```

For a full experiment, load STL-10 images as NumPy arrays and use the training/evaluation functions from the `src/` package.
